# 05 — Split freeze and structural gates

**Objective.** Freeze all block IDs, expanding folds, background/audit case IDs, endpoint count gates, preregistration, and the Phase-0 protocol lock.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

No model result is permitted to determine cohort, endpoint, feature, split, background, or local-audit inclusion.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("05", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import shutil
from cruxvc.explanations import sample_background_ids, sample_local_audit_cases
from cruxvc.hashing import sha256_file
from cruxvc.io import read_json, read_table, write_json, write_table
from cruxvc.manifest import create_protocol_lock, signing_key_from_environment
from cruxvc.splitting import assign_time_blocks, block_count_table, expanding_window_folds, structural_endpoint_gates
from cruxvc.validation import assert_expected_counts, assert_no_split_leakage

cohort_path = P.processed / "cohort_labels.parquet"
strict_path = P.processed / "features_strict.parquet"
CTX.recorder.inputs.extend([cohort_path, strict_path, P.protocol / "feature_time_ledger.csv"])
cohort = read_table(cohort_path)
strict = read_table(strict_path)

In [ ]:
split_ids = assign_time_blocks(cohort, CFG["cohort"]["split_blocks"])
assert_no_split_leakage(split_ids)
development = cohort[
    cohort["case_id"].isin(
        split_ids.loc[split_ids["time_block"].astype(str).eq("development"), "case_id"]
    )
]
folds = expanding_window_folds(development)
outcomes = ["F18", "F36", "B+36", "C+36", "A36"]
counts = block_count_table(cohort, split_ids, outcomes)
source_manifest = read_json(P.protocol / "source_manifest.json")
strict_expected = bool(
    source_manifest["all_expected_hashes_match"]
    and CFG["execution"]["strict_expected_counts_when_hashes_match"]
)
count_differences = assert_expected_counts(
    counts,
    CFG["expected_block_counts"],
    strict=strict_expected,
)
gates = structural_endpoint_gates(counts, outcomes, CFG["endpoint_gates"])
confirmatory = gates[gates["outcome"].isin(CFG["outcomes"]["confirmatory"])]
if not confirmatory["structural_pass"].all():
    raise RuntimeError(
        "A confirmatory endpoint failed structural gates:"
        + chr(10) + confirmatory.to_string(index=False)
    )

In [ ]:
split_path = write_table(split_ids, P.protocol / "split_ids.parquet")
folds_path = write_table(folds, P.protocol / "development_folds.parquet")
counts_path = write_table(counts, P.audits / "05_block_outcome_counts.csv")
gates_path = write_table(gates, P.audits / "05_structural_endpoint_gates.csv")

full_profile = CFG["compute_profiles"]["full"]
strict_dedup = strict.drop(columns=[c for c in strict.columns if c in cohort.columns and c not in ("case_id", "company_permalink", "t0")])
analysis_frame = strict_dedup.merge(cohort, on=["case_id", "company_permalink", "t0"], how="inner").merge(
    split_ids[["case_id", "time_block"]], on="case_id", how="left"
)
development_frame = analysis_frame[analysis_frame["time_block"].astype(str).eq("development")]
test_frame = analysis_frame[analysis_frame["time_block"].astype(str).eq("final_test")]
background_ids = sample_background_ids(
    development_frame,
    n_sets=int(full_profile["background_sets"]),
    n_per_set=int(full_profile["background_n"]),
    seed=int(CFG["execution"]["random_seed"]),
)
audit_cases = sample_local_audit_cases(
    test_frame,
    n=int(full_profile["local_audit_n"]),
    outcome_columns=CFG["outcomes"]["confirmatory"],
    seed=int(CFG["execution"]["random_seed"]) + 1,
)
background_path = write_table(background_ids, P.protocol / "explanation_background_ids.csv")
audit_cases_path = write_table(
    audit_cases[[
        "case_id", "landmark_round_type", "joint_outcome_pattern", "audit_stratum",
        "design_weight", "audit_inclusion_probability",
    ]],
    P.protocol / "local_audit_case_ids.csv",
)

In [ ]:
prereg_source = P.protocol / "preregistration_template.yaml"
prereg_frozen = P.locks / "preregistration_frozen.yaml"
shutil.copy2(prereg_source, prereg_frozen)
lock_payload = {
    "lock_type": "phase0",
    "protocol_version": "2.2",
    "source_manifest_sha256": sha256_file(P.protocol / "source_manifest.json"),
    "cohort_sha256": sha256_file(cohort_path),
    "strict_features_sha256": sha256_file(strict_path),
    "feature_ledger_sha256": sha256_file(P.protocol / "feature_time_ledger.csv"),
    "split_ids_sha256": sha256_file(split_path),
    "folds_sha256": sha256_file(folds_path),
    "counts_sha256": sha256_file(counts_path),
    "gates_sha256": sha256_file(gates_path),
    "background_ids_sha256": sha256_file(background_path),
    "audit_case_ids_sha256": sha256_file(audit_cases_path),
    "preregistration_sha256": sha256_file(prereg_frozen),
    "count_differences": count_differences,
    "final_test_model_results_opened": False,
}
phase0_lock = create_protocol_lock(
    lock_payload,
    P.locks / "phase0_lock.json",
    author_email=CFG["project"]["author_email"],
    signing_key=signing_key_from_environment(),
)

In [ ]:
import shutil
from pathlib import Path
m = Path("/content/drive/MyDrive/CRUX_Research/crux-vc/results/manifests/05_split_freeze_and_structural_gates.json")
shutil.move(m, m.with_name("05_split_freeze_and_structural_gates.superseded_unsigned_lock.json"))
print("prior manifest archived")

In [ ]:
CTX.recorder.complete([
    split_path, folds_path, counts_path, gates_path, background_path, audit_cases_path, prereg_frozen, phase0_lock
])
print(counts.to_string(index=False))
print(gates.to_string(index=False))